In [ ]:
# 导入 pandas 库，主要用于读取 CSV、处理表格数据
# 通常简写为 pd
import pandas as pd

# 导入 numpy 库，主要用于数值计算
# 当前这段代码暂时没有大量使用 numpy，但后续建模会用到
import numpy as np


# 读取训练集数据
# "../" 表示返回上一级文件夹
# 例如你的 notebook 在 notebooks/ 文件夹里，
# 数据在 data/raw/ 文件夹里，
# 所以路径要写成 ../data/raw/train_format2.csv
train = pd.read_csv("../data/raw/train_format2.csv")


# 读取测试集数据
# 测试集一般没有 label，或者 label 为空
# 后续模型训练完成后，会对 test 进行预测
test = pd.read_csv("../data/raw/test_format2.csv")


# 输出训练集的形状
# shape 会返回：行数、列数
# 例如 (7030723, 50) 表示有 7030723 行、50 列
print("train shape:", train.shape)


# 输出测试集的形状
# 用来确认测试集有多少行、多少列
print("test shape:", test.shape)


# 查看训练集前 5 行
# 目的是快速看一下字段长什么样、数据格式是否正常
display(train.head())


# 查看测试集前 5 行
# 用来和训练集对比字段是否一致
display(test.head())


# 查看训练集字段信息
# 包括：
# 1. 每一列的列名
# 2. 每一列有多少非空值
# 3. 每一列的数据类型
# 4. 整个 DataFrame 占用内存
print(train.info())


# 查看测试集字段信息
# 主要用来检查是否存在缺失值、字段类型是否正常
print(test.info())


# 输出训练集的所有字段名
# 方便我们判断哪些是用户特征、商家特征、行为特征、标签字段
print(train.columns.tolist())


# 输出测试集的所有字段名
# 后续要和训练集字段对齐
print(test.columns.tolist())


# 查看训练集 label 的数量分布
# label 是我们要预测的目标变量
# 一般：
# label = 1 表示未来会复购
# label = 0 表示未来不会复购
# 如果有 label = -1，通常需要先剔除，不能直接作为二分类标签
print(train["label"].value_counts(dropna=False))


# 查看训练集 label 的比例分布
# normalize=True 表示输出比例，而不是数量
# 例如：
# 0    0.94
# 1    0.06
# 说明 94% 是未复购，6% 是复购
# 这可以判断是否存在样本不均衡问题
print(train["label"].value_counts(normalize=True, dropna=False))

train shape: (7030723, 6)
test shape: (7027943, 6)


,user_id,age_range,gender,merchant_id,label,activity_log
0,34176,6.0,0.0,944,-1,408895:1505:7370:1107:0
1,34176,6.0,0.0,412,-1,17235:1604:4396:0818:0#954723:1604:4396:0818:0...
2,34176,6.0,0.0,1945,-1,231901:662:2758:0818:0#231901:662:2758:0818:0#...
3,34176,6.0,0.0,4752,-1,174142:821:6938:1027:0
4,34176,6.0,0.0,643,-1,716371:1505:968:1024:3


,user_id,age_range,gender,merchant_id,label,activity_log
0,163968,0.0,0.0,4378,-1.0,101206:812:6968:0614:0
1,163968,0.0,0.0,2300,-1.0,588758:844:3833:0618:0#71782:844:3833:1111:2#7...
2,163968,0.0,0.0,1551,-1.0,312747:243:1954:0627:0#312747:243:1954:0627:0#...
3,163968,0.0,0.0,4343,-1.0,932390:1612:3201:0628:0
4,163968,0.0,0.0,4911,-1.0,957657:662:3089:0612:0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7030723 entries, 0 to 7030722
Data columns (total 6 columns):
 #   Column        Dtype  
---  ------        -----  
 0   user_id       int64  
 1   age_range     float64
 2   gender        float64
 3   merchant_id   int64  
 4   label         int64  
 5   activity_log  object 
dtypes: float64(2), int64(3), object(1)
memory usage: 321.8+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7027943 entries, 0 to 7027942
Data columns (total 6 columns):
 #   Column        Dtype  
---  ------        -----  
 0   user_id       int64  
 1   age_range     float64
 2   gender        float64
 3   merchant_id   int64  
 4   label         float64
 5   activity_log  object 
dtypes: float64(3), int64(2), object(1)
memory usage: 321.7+ MB
None
['user_id', 'age_range', 'gender', 'merchant_id', 'label', 'activity_log']
['user_id', 'age_range', 'gender', 'merchant_id', 'label', 'activity_log']
label
-1    6769859
 0     244912
 1      15952
Name: co

In [2]:
# 只保留 label 为 0 和 1 的样本
# label=-1 暂时不参与二分类建模
train_model = train[train["label"].isin([0, 1])].copy()

# 先抽样 5 万行，测试代码逻辑
# random_state=42 是为了保证每次抽样结果一致
sample = train_model.sample(n=50000, random_state=42)

print("train_model shape:", train_model.shape)
print("sample shape:", sample.shape)

print(sample["label"].value_counts())
print(sample["label"].value_counts(normalize=True))

train_model shape: (260864, 6)
sample shape: (50000, 6)
label
0    46972
1     3028
Name: count, dtype: int64
label
0    0.93944
1    0.06056
Name: proportion, dtype: float64


In [3]:
def extract_activity_features(activity_log):
    """
    输入：一行 activity_log 字符串
    输出：这一行对应的行为特征
    
    activity_log 格式：
    item_id:cat_id:brand_id:time_stamp:action_type#item_id:cat_id:brand_id:time_stamp:action_type
    """
    
    # 如果 activity_log 是缺失值，返回全 0 特征
    if pd.isna(activity_log):
        return pd.Series({
            "total_actions": 0,
            "click_cnt": 0,
            "cart_cnt": 0,
            "buy_cnt": 0,
            "fav_cnt": 0,
            "unique_items": 0,
            "unique_cats": 0,
            "unique_brands": 0,
            "unique_days": 0
        })
    
    # 按 # 拆分成多条行为记录
    records = str(activity_log).split("#")
    
    # 用来存储每条行为拆出来的信息
    item_list = []
    cat_list = []
    brand_list = []
    day_list = []
    action_list = []
    
    for record in records:
        # 每条行为理论上应该由 5 个部分组成
        parts = record.split(":")
        
        # 如果不是 5 个部分，说明这一条异常，跳过
        if len(parts) != 5:
            continue
        
        item_id, cat_id, brand_id, time_stamp, action_type = parts
        
        item_list.append(item_id)
        cat_list.append(cat_id)
        brand_list.append(brand_id)
        day_list.append(time_stamp)
        action_list.append(action_type)
    
    # 统计不同类型行为数量
    click_cnt = action_list.count("0")
    cart_cnt = action_list.count("1")
    buy_cnt = action_list.count("2")
    fav_cnt = action_list.count("3")
    
    return pd.Series({
        # 总行为次数
        "total_actions": len(action_list),
        
        # 点击次数
        "click_cnt": click_cnt,
        
        # 加购次数
        "cart_cnt": cart_cnt,
        
        # 购买次数
        "buy_cnt": buy_cnt,
        
        # 收藏次数
        "fav_cnt": fav_cnt,
        
        # 互动过的商品数量
        "unique_items": len(set(item_list)),
        
        # 互动过的品类数量
        "unique_cats": len(set(cat_list)),
        
        # 互动过的品牌数量
        "unique_brands": len(set(brand_list)),
        
        # 活跃天数
        "unique_days": len(set(day_list))
    })

In [4]:
# 对 sample 的 activity_log 进行特征提取
activity_features = sample["activity_log"].apply(extract_activity_features)

# 把提取出来的特征和原始字段合并
sample_features = pd.concat(
    [
        sample[["user_id", "age_range", "gender", "merchant_id", "label"]].reset_index(drop=True),
        activity_features.reset_index(drop=True)
    ],
    axis=1
)

display(sample_features.head())
print(sample_features.shape)

,user_id,age_range,gender,merchant_id,label,total_actions,click_cnt,cart_cnt,buy_cnt,fav_cnt,unique_items,unique_cats,unique_brands,unique_days
0,399620,3.0,1.0,310,0,9,8,0,1,0,4,3,1,1
1,183656,2.0,1.0,3129,0,3,2,0,1,0,2,1,1,1
2,214005,0.0,0.0,3609,0,12,9,0,1,2,2,1,1,5
3,76778,0.0,1.0,2824,0,3,2,0,1,0,2,1,1,1
4,258043,0.0,0.0,4760,1,3,2,0,1,0,1,1,1,1


(50000, 14)


In [5]:
# 防止除以 0 的函数
def safe_divide(a, b):
    return a / (b + 1e-6)

# 购买 / 点击，表示点击后的购买转化倾向
sample_features["buy_click_rate"] = safe_divide(
    sample_features["buy_cnt"], 
    sample_features["click_cnt"]
)

# 加购 / 点击，表示用户对该商家的潜在购买意愿
sample_features["cart_click_rate"] = safe_divide(
    sample_features["cart_cnt"], 
    sample_features["click_cnt"]
)

# 收藏 / 点击，表示用户对该商家的兴趣强度
sample_features["fav_click_rate"] = safe_divide(
    sample_features["fav_cnt"], 
    sample_features["click_cnt"]
)

# 购买 / 总行为，表示用户行为中的购买占比
sample_features["buy_action_rate"] = safe_divide(
    sample_features["buy_cnt"], 
    sample_features["total_actions"]
)

display(sample_features.head())

,user_id,age_range,gender,merchant_id,label,total_actions,click_cnt,cart_cnt,buy_cnt,fav_cnt,unique_items,unique_cats,unique_brands,unique_days,buy_click_rate,cart_click_rate,fav_click_rate,buy_action_rate
0,399620,3.0,1.0,310,0,9,8,0,1,0,4,3,1,1,0.125000,0.0,0.000000,0.111111
1,183656,2.0,1.0,3129,0,3,2,0,1,0,2,1,1,1,0.500000,0.0,0.000000,0.333333
2,214005,0.0,0.0,3609,0,12,9,0,1,2,2,1,1,5,0.111111,0.0,0.222222,0.083333
3,76778,0.0,1.0,2824,0,3,2,0,1,0,2,1,1,1,0.500000,0.0,0.000000,0.333333
4,258043,0.0,0.0,4760,1,3,2,0,1,0,1,1,1,1,0.500000,0.0,0.000000,0.333333


In [9]:

# 从 sklearn 中导入 train_test_split
# 作用：把我们已有的数据拆分成“训练集”和“验证集”
# 训练集：用来让模型学习规律
# 验证集：用来测试模型在没见过的数据上的表现
from sklearn.model_selection import train_test_split

# 从 sklearn 中导入常用的分类模型评估指标
# roc_auc_score：计算 AUC，用来衡量模型区分复购用户和非复购用户的能力
# f1_score：计算 F1，用来综合衡量 Precision 和 Recall
# precision_score：计算 Precision，预测为复购的人里面，有多少真的复购
# recall_score：计算 Recall，真实复购的人里面，有多少被模型找出来
# confusion_matrix：混淆矩阵，用来查看 TP、FP、TN、FN
# classification_report：分类报告，一次性输出 precision、recall、f1-score 等指标
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report

# 从 sklearn 中导入随机森林分类模型
# RandomForestClassifier 是一个常用的机器学习分类模型
# 它由很多棵决策树组成，通过多棵树投票来判断用户是否会复购
# 这里我们用它作为第一个基础模型，因为它不太需要复杂的数据标准化，比较适合快速跑通流程
from sklearn.ensemble import RandomForestClassifier


# y 是模型要预测的目标变量，也就是 label
# 在这个项目里：
# label = 1 表示用户未来会复购
# label = 0 表示用户未来不会复购
#
# 注意：
# 你前面已经筛掉了 label = -1 的数据
# 所以 sample_features 里理论上只剩 0 和 1
#
# astype(int) 的作用：
# 把 label 转成整数类型，避免后续模型训练时因为数据类型问题报错
y = sample_features["label"].astype(int)


# X 是模型用来学习的特征变量
# 也就是除了 label 以外，所有可以帮助模型判断用户是否复购的信息
#
# 我们这里从 sample_features 中删除三列：
# 1. user_id：用户编号
# 2. merchant_id：商家编号
# 3. label：真实答案
#
# 为什么删除 user_id 和 merchant_id？
# 因为 user_id 和 merchant_id 只是编号，不是有业务含义的行为特征。
# 如果直接放进模型，模型可能会“记住某些用户编号或商家编号”，
# 而不是学习真正的行为规律。
#
# 为什么删除 label？
# 因为 label 是答案，不能作为输入特征。
# 如果把答案也放进 X，模型就相当于作弊，评估结果会虚高。
X = sample_features.drop(columns=["user_id", "merchant_id", "label"])


# fillna(0) 的作用：
# 把缺失值 NaN 填充为 0
#
# 为什么要填充缺失值？
# 大多数 sklearn 模型不能直接处理 NaN。
# 如果 X 里面有空值，模型训练时可能会报错。
#
# 为什么这里用 0 填充？
# 在这个项目里，大多数行为类特征如果缺失，可以先理解为没有发生该行为。
# 例如：
# click_cnt 缺失，可以先填 0
# cart_cnt 缺失，可以先填 0
# buy_cnt 缺失，可以先填 0
#
# 这是一种基础处理方法，后续如果追求更严谨，可以按字段含义分别处理。
X = X.fillna(0)

# train_test_split 用来把数据拆成两部分：
# 1. X_train, y_train：训练集，用来训练模型
# 2. X_valid, y_valid：验证集，用来评估模型效果
#
# 参数解释：
#
# X：
# 输入特征，例如点击次数、购买次数、活跃天数、转化率等
#
# y：
# 真实标签，也就是是否复购
#
# test_size=0.2：
# 表示 20% 的数据作为验证集，80% 的数据作为训练集
#
# random_state=42：
# 固定随机种子，保证每次拆分结果一致
# 这样你每次运行代码，结果不会随机变化太大
#
# stratify=y：
# 按照 y 的比例进行分层抽样
# 因为你的复购样本非常少，label=1 占比很低
# 如果不加 stratify，可能验证集中正样本比例异常，导致评估不稳定
#
# 举例：
# 如果原始数据中 1 的比例是 6%，0 的比例是 94%
# 加上 stratify=y 后，训练集和验证集也会大致保持这个比例
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 创建一个随机森林分类模型
# 这个模型会学习 X_train 中的行为特征和 y_train 中的复购标签之间的关系
model = RandomForestClassifier(

    # n_estimators=200：
    # 随机森林里有 200 棵决策树
    # 树越多，模型通常越稳定，但训练时间也会更长
    n_estimators=200,

    # max_depth=8：
    # 限制每棵树最多长到 8 层
    # 如果树太深，模型容易把训练数据“背下来”，造成过拟合
    # 设置 max_depth 可以让模型更稳健
    max_depth=8,

    # min_samples_leaf=20：
    # 每个叶子节点至少要有 20 个样本
    # 叶子节点可以理解为模型最终做判断的末端规则
    # 这个值越大，模型越保守，不容易过拟合
    min_samples_leaf=20,

    # class_weight="balanced"：
    # 处理样本不均衡问题
    # 你的数据里复购用户 label=1 很少
    # 如果不加这个参数，模型可能倾向于全部预测为 0
    # balanced 会给少数类 label=1 更高权重，让模型更重视复购样本
    class_weight="balanced",

    # random_state=42：
    # 固定随机种子，保证每次训练结果可复现
    random_state=42,

    # n_jobs=-1：
    # 使用电脑所有可用 CPU 核心进行训练
    # 可以加快训练速度
    n_jobs=-1
)

# 用训练集训练模型
# X_train 是输入特征
# y_train 是真实标签
#
# 模型会在这里学习：
# 哪些行为特征更可能对应复购
# 哪些行为特征更可能对应不复购
#
# 例如它可能学到：
# buy_cnt 越高，复购概率可能越高
# cart_click_rate 越高，复购概率可能越高
# unique_days 越多，用户活跃程度越高
model.fit(X_train, y_train)

# predict_proba 用来输出每个样本属于不同类别的概率
#
# 对于二分类问题，输出通常有两列：
# 第 1 列：预测为 0 的概率，也就是不复购的概率
# 第 2 列：预测为 1 的概率，也就是复购的概率
#
# 例如某一行输出：
# [0.82, 0.18]
# 表示模型认为：
# 不复购概率 = 82%
# 复购概率 = 18%
#
# [:, 1] 表示取第 2 列，也就是复购概率
valid_prob = model.predict_proba(X_valid)[:, 1]

# 查看前 10 个用户-商家样本的复购概率
print(valid_prob[:10])

[0.50007367 0.47976981 0.42575937 0.40073342 0.53759488 0.42885952
 0.40449977 0.50857059 0.3801046  0.48151136]


In [8]:
# 模型输出的是概率，例如：
# 0.12、0.38、0.67
#
# 但是 F1、Precision、Recall 这些指标需要 0/1 预测结果
# 所以我们要设置一个阈值 threshold
#
# 这里使用 0.5 作为默认阈值：
# 如果复购概率 >= 0.5，就预测为 1，也就是会复购
# 如果复购概率 < 0.5，就预测为 0，也就是不会复购
#
# astype(int)：
# 把 True/False 转成 1/0
valid_pred = (valid_prob >= 0.5).astype(int)

# 计算 AUC
#
# AUC 用来衡量模型区分正负样本的能力
# 也就是模型能不能把真正会复购的用户排在更靠前的位置
#
# AUC 取值范围一般是 0 到 1：
# 0.5 左右：模型基本等于随机猜
# 0.6-0.7：有一定区分能力
# 0.7-0.8：区分能力较好
# 0.8 以上：区分能力较强
#
# 注意：
# AUC 使用的是 valid_prob 概率，而不是 valid_pred 0/1 结果
print("AUC:", roc_auc_score(y_valid, valid_prob))

# 计算 F1
#
# F1 是 Precision 和 Recall 的综合指标
# 它适合在样本不均衡时使用
#
# F1 越高，说明模型在“预测准确”和“覆盖正样本”之间平衡得越好
#
# F1 使用的是 valid_pred，也就是 0/1 预测结果
print("F1:", f1_score(y_valid, valid_pred))

# 计算 Precision，中文通常叫“精确率”
#
# Precision 的含义：
# 在模型预测为“会复购”的用户中，真正复购的比例是多少
#
# 业务理解：
# 如果 Precision 高，说明你圈出来的高潜用户比较准
# 适合用于优惠券成本较高、触达资源有限的场景
#
# 例如：
# 模型预测 1000 个用户会复购
# 其中实际有 200 个真的复购
# Precision = 200 / 1000 = 20%
print("Precision:", precision_score(y_valid, valid_pred))

# 计算 Recall，中文通常叫“召回率”
#
# Recall 的含义：
# 在所有真实会复购的用户中，模型成功识别出来的比例是多少
#
# 业务理解：
# 如果 Recall 高，说明模型尽可能找出了更多潜在复购用户
# 适合用于低成本触达场景，例如站内消息、低成本内容推荐
#
# 例如：
# 真实复购用户有 1000 个
# 模型成功识别出 600 个
# Recall = 600 / 1000 = 60%
print("Recall:", recall_score(y_valid, valid_pred))

# 输出混淆矩阵
#
# confusion_matrix 会输出一个 2x2 的矩阵：
#
# [[TN, FP],
#  [FN, TP]]
#
# TN：真实不复购，模型也预测不复购
# FP：真实不复购，但模型预测复购，也叫误报
# FN：真实复购，但模型预测不复购，也叫漏报
# TP：真实复购，模型也预测复购
#
# 对这个项目来说：
# FP 太多：说明你给很多不会复购的人发了券，浪费运营资源
# FN 太多：说明很多真正会复购的人没被识别出来，错过运营机会
print("Confusion Matrix:")
print(confusion_matrix(y_valid, valid_pred))

# classification_report 会输出每个类别的：
# precision
# recall
# f1-score
# support
#
# support 表示每个类别真实样本数量
#
# 对你来说，重点看 label=1 的指标
# 因为 label=1 才是复购用户，是运营真正关心的人群
print(classification_report(y_valid, valid_pred))

AUC: 0.6314397540456622
F1: 0.17101353582040277
Precision: 0.10689228229467602
Recall: 0.4273927392739274
Confusion Matrix:
[[7230 2164]
 [ 347  259]]
              precision    recall  f1-score   support

           0       0.95      0.77      0.85      9394
           1       0.11      0.43      0.17       606

    accuracy                           0.75     10000
   macro avg       0.53      0.60      0.51     10000
weighted avg       0.90      0.75      0.81     10000

